In [ ]:
import pandas as pd
import numpy as np
import os
print("Libraries successfully imported!")

In [ ]:
matches=pd.read_csv("../data/processed/epl_matches_clean.csv")
matches.dtypes
matches["Date"]=pd.to_datetime(matches["Date"], errors="coerce")
matches.dtypes
matches = matches.dropna(
    subset=["Date", "HomeTeam", "AwayTeam", "HomeGoals", "AwayGoals", "MatchResult"]
).copy()
matches=matches.sort_values("Date").reset_index(drop=True)
matches["MatchID"]=range(len(matches))
matches.head()


In [ ]:
# Convert matches into team history
# homeTeam
home_history=matches[["MatchID", "Date", "HomeTeam", "AwayTeam", "HomeGoals", "AwayGoals"]].copy()
home_history.columns=["MatchID", "Date", "Team", "Opponent", "GoalsFor", "GoalsAgainst"]
# awayTeam
away_history=matches[["MatchID", "Date", "AwayTeam", "HomeTeam", "AwayGoals", "HomeGoals"]].copy()
away_history.columns=["MatchID", "Date", "Team", "Opponent", "GoalsFor", "GoalsAgainst"]
#combining
team_history=pd.concat(
    [home_history, away_history],
    ignore_index=True
)
#match points
team_history["Points"]=np.select(
    [
        team_history["GoalsFor"]>team_history["GoalsAgainst"],
        team_history["GoalsFor"]==team_history["GoalsAgainst"]
    ],
    [3, 1],
    default=0
)
team_history["Win"]=(team_history["GoalsFor"]>team_history["GoalsAgainst"]).astype(int)
team_history["Draw"]=(team_history["GoalsFor"]==team_history["GoalsAgainst"]).astype(int)
team_history["Loss"]=(team_history["GoalsFor"]<team_history["GoalsAgainst"]).astype(int)
#Sort each teams matches
team_history=team_history.sort_values(["Team", "Date", "MatchID"]).reset_index(drop=True)
team_history.head(10)

In [ ]:
# Columns we want to calculate historical averages for
history_columns = [
    "Points",
    "Win",
    "GoalsFor",
    "GoalsAgainst"
]
# Calculate features using only previous matches
for column in history_columns:
    team_history[f"Recent{column}"] = (
        team_history
        .groupby("Team")[column]
        .transform(
            lambda s: s.shift(1).rolling(
                window=5,
                min_periods=1
            ).mean()
        )
    )
# Rename features to clearer names
team_history = team_history.rename(columns={
    "RecentPoints": "RecentPoints",
    "RecentWin": "RecentWinRate",
    "RecentGoalsFor": "RecentGoalsScored",
    "RecentGoalsAgainst": "RecentGoalsConceded"
})
# Calculate recent goal difference
team_history["RecentGoalDifference"] = (
    team_history["RecentGoalsScored"]
    - team_history["RecentGoalsConceded"]
)
team_history[
    [
        "Team",
        "Date",
        "Points",
        "RecentPoints",
        "RecentWinRate",
        "RecentGoalsScored",
        "RecentGoalsConceded",
        "RecentGoalDifference"
    ]
].tail(15)

In [ ]:
#No of matches played before current match
team_history["MatchesPlayedBefore"]=(team_history.groupby("Team").cumcount())
#Days since last match
team_history["DaysSinceLastMatch"]=(team_history.groupby("Team")["Date"].diff().dt.days)
team_history[["Team", "Date", "MatchesPlayedBefore", "DaysSinceLastMatch"]].tail(5)

In [ ]:
# Attach each team history to its actual home/away match role.
home_features = team_history.merge(
    matches[["MatchID", "HomeTeam"]], on="MatchID", how="inner"
).query("Team == HomeTeam")[["MatchID"] + feature_columns].copy()
home_features = home_features.rename(columns={c: f"Home{c}" for c in feature_columns})

away_features = team_history.merge(
    matches[["MatchID", "AwayTeam"]], on="MatchID", how="inner"
).query("Team == AwayTeam")[["MatchID"] + feature_columns].copy()
away_features = away_features.rename(columns={c: f"Away{c}" for c in feature_columns})

feature_matches = matches.merge(home_features, on="MatchID", how="left", validate="one_to_one")
feature_matches = feature_matches.merge(away_features, on="MatchID", how="left", validate="one_to_one")
print("Feature dataset shape:", feature_matches.shape)
feature_matches.head()


In [ ]:
#Check data leakage
leakage_columns=["HomeGoals", "AwayGoals", "FullTimeScore", "TotalGoals", "GoalDifference", "MatchResult"]
existing_leakage_columns=[
    column for column in leakage_columns
    if column in feature_matches.columns
]
print(existing_leakage_columns)
history_features=[column for column in feature_matches.columns
                  if "Recent" in column
                  or "MatchesPlayedBefore" in column
                  or "DaysSinceLastMatch" in column]
print(history_features)
#Check missing values
feature_matches[history_features].isnull().sum()

In [ ]:
#Save engineered features
os.makedirs("../data/processed", exist_ok=True)
feature_matches.to_csv("../data/processed/epl_matches_features.csv", index=False)
print("Feature dataset saved successfully!")